# Day 34 — Transfer learning: frozen backbone + new head

Status: COMPLETE. MobileNetV2 (ImageNet) frozen, 2,562-param head trained on
cached 1280-dim features (15 epochs, seconds each after one cache pass).
9/9 tests pass.

In [1]:
import json
from pathlib import Path

m = json.loads(Path("../models/transfer_metrics.json").read_text())
print(f"frozen transfer: accuracy {m['accuracy']} | ROC-AUC {m['roc_auc']} | "
      f"PR-AUC {m['pr_auc']}")
print(f"F1 {m['f1']} | precision {m['precision']}, recall {m['recall']}")
c = m["confusion"]
print(f"confusion: tn={c['tn']} fp={c['fp']} fn={c['fn']} tp={c['tp']}")
print("vs scratch baseline: 0.855 -> 0.958 ROC-AUC (+0.10), recall 0.79 -> 0.97")
print("model: models/mobilenet_frozen.pt | cache: data/cache_mobilenet/ "
      "(warm reruns skip CNN)")

frozen transfer: accuracy 0.8638 | ROC-AUC 0.9584 | PR-AUC 0.9735
F1 0.8992 | precision 0.8366, recall 0.9718
confusion: tn=160 fp=74 fn=11 tp=379
vs scratch baseline: 0.855 -> 0.958 ROC-AUC (+0.10), recall 0.79 -> 0.97
model: models/mobilenet_frozen.pt | cache: data/cache_mobilenet/ (warm reruns skip CNN)


## Findings

1. **+0.10 ROC-AUC from pretrained features.** The head-only model sees the same
   4,695 images as the baseline but starts from ImageNet edges/textures — the
   Day-33 prediction confirmed numerically.
2. **Recall 0.97 at precision 0.84** — only 11 missed pneumonias. For triage
   (flag-for-review), this operating direction is right; threshold tuning
   (Day 36) refines it.
3. **Backbone freezing is what made CPU training feasible**: one slow cache pass,
   then 15 head epochs in seconds. Unfreezing everything on CPU would take hours
   and risk destroying the pretrained filters — hence gradual unfreeze (Day 35).

## Implementation note (caught by testing, not by metrics)

MobileNetV2's global pooling lives in `forward()`, not `.features` — caching
raw backbone output produced 62,720-dim vectors and a shape crash. The cache
now mirrors forward exactly (pool + flatten → 1280). Abstractions leak; verify
tensor shapes at every pipeline joint.